In [ ]:
!pip install biopython

In [ ]:
from Bio.PDB import PDBParser, PDBIO, Superimposer, Select, Polypeptide
from Bio.PDB.Polypeptide import PPBuilder
from Bio import pairwise2
from Bio.Align import substitution_matrices
from Bio.Data.IUPACData import protein_letters_3to1
import numpy as np
import os


In [ ]:
# -------------------- Utility Classes --------------------

class ChainSelect(Select):
    def __init__(self, target_chain_id):
        self.target_chain_id = target_chain_id
    def accept_chain(self, chain):
        return chain.id == self.target_chain_id

class KeepOnlyDrug(Select):
    def __init__(self, drug_resname):
        self.drug_resname = drug_resname
    def accept_residue(self, residue):
        hetfield, _, _ = residue.get_id()
        return hetfield == ' ' or residue.get_resname() == self.drug_resname

# -------------------- Preprocessing --------------------

def remove_chains(chain_id, pdb_file, out_file):
    structure = PDBParser(QUIET=True).get_structure('struct', pdb_file)
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_file, ChainSelect(chain_id))

def remove_ligands(pdb_file, drug_resname, out_file):
    structure = PDBParser(QUIET=True).get_structure('struct', pdb_file)
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_file, KeepOnlyDrug(drug_resname))

def rename_chains_and_ligands(pdb_file, chain_map, ligand_map, out_file):
    structure = PDBParser(QUIET=True).get_structure('struct', pdb_file)
    for model in structure:
        for chain in model:
            if chain.id in chain_map:
                chain.id = chain_map[chain.id]
            for residue in chain:
                hetfield, _, _ = residue.get_id()
                if hetfield != ' ' and residue.get_resname() in ligand_map:
                    residue.resname = ligand_map[residue.get_resname()]
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_file)

# -------------------- Sequence & Atom Extraction --------------------

def extract_sequence_and_atoms(structure, chain_id):
    ppb = PPBuilder()
    sequence = ""
    atoms = []
    for model in structure:
        for chain in model:
            if chain.id == chain_id:
                for pp in ppb.build_peptides(chain):
                    for res, aa in zip(pp, str(pp.get_sequence())):
                        if "CA" in res:
                            sequence += aa
                            atoms.append(res["CA"])
    return sequence, atoms

def extract_sequence_and_coords(chain):
    seq, coords, residues = [], [], []
    for res in chain:
        if Polypeptide.is_aa(res, standard=True) and "CA" in res:
            seq.append(res.resname)
            coords.append(res["CA"])
            residues.append(res)
    one_letter_seq = ''.join([protein_letters_3to1.get(r.get_resname().capitalize(), 'X') for r in residues])
    return one_letter_seq, coords

# -------------------- Structure Saving --------------------

def save_combined_structure(struct1, struct2, chain1_id, chain2_id, output_path):
    io = PDBIO()
    io.set_structure(struct1)
    io.save("temp1.pdb", ChainSelect(chain1_id))
    io.set_structure(struct2)
    io.save("temp2.pdb", ChainSelect(chain2_id))
    with open("temp1.pdb") as f1, open("temp2.pdb") as f2, open(output_path, "w") as fout:
        fout.write(f1.read())
        fout.write(f2.read())
        fout.write("END\n")
    os.remove("temp1.pdb")
    os.remove("temp2.pdb")


In [ ]:

# -------------------- Alignment --------------------

def align_structures_local(pdb1_path, pdb2_path, chain1_id='A', chain2_id='A', output_path='aligned_complex.pdb'):
    parser = PDBParser(QUIET=True)
    struct1 = parser.get_structure("ref", pdb1_path)
    struct2 = parser.get_structure("mov", pdb2_path)

    seq1, atoms1 = extract_sequence_and_atoms(struct1, chain1_id)
    seq2, atoms2 = extract_sequence_and_atoms(struct2, chain2_id)

    if not seq1 or not seq2:
        print("Sequence extraction failed.")
        return None

    matrix = substitution_matrices.load("BLOSUM62")
    alignment = pairwise2.align.localds(seq1, seq2, matrix, -10, -0.5, one_alignment_only=True)
    if not alignment:
        print("No alignment found.")
        return None

    a1, a2 = alignment[0].seqA, alignment[0].seqB
    matched_atoms1, matched_atoms2, i1, i2 = [], [], 0, 0

    for r1, r2 in zip(a1, a2):
        if r1 != '-' and r2 != '-':
            try:
                matched_atoms1.append(atoms1[i1])
                matched_atoms2.append(atoms2[i2])
            except IndexError:
                print("Index mismatch during atom extraction.")
                break
        if r1 != '-': i1 += 1
        if r2 != '-': i2 += 1

    if len(matched_atoms1) < 3:
        print("Too few atoms to align.")
        return None

    sup = Superimposer()
    sup.set_atoms(matched_atoms1, matched_atoms2)
    sup.apply(struct2.get_atoms())

    save_combined_structure(struct1, struct2, chain1_id, chain2_id, output_path)
    print(f"RMSD (local): {sup.rms:.3f} Å — saved to {output_path}")
    return sup.rms

def get_aligned_coords(seq1, coords1, seq2, coords2):
    matrix = substitution_matrices.load("BLOSUM62")
    alignment = pairwise2.align.localds(seq1, seq2, matrix, -11, -1, one_alignment_only=True)[0]
    a1, a2 = alignment.seqA, alignment.seqB
    aligned1, aligned2, i1, i2 = [], [], 0, 0

    for r1, r2 in zip(a1, a2):
        if r1 != '-' and r2 != '-':
            aligned1.append(coords1[i1])
            aligned2.append(coords2[i2])
        if r1 != '-': i1 += 1
        if r2 != '-': i2 += 1

    return aligned1, aligned2

def iterative_alignment(fixed_coords, moving_coords, max_iter=10, cutoff=4.0):
    sup = Superimposer()
    for i in range(max_iter):
        sup.set_atoms(fixed_coords, moving_coords)
        sup.apply(moving_coords)
        dists = np.linalg.norm(np.array([a.get_coord() for a in fixed_coords]) -
                               np.array([a.get_coord() for a in moving_coords]), axis=1)
        indices = np.where(dists <= cutoff)[0]
        if len(indices) == len(fixed_coords):
            print(f"Converged after {i+1} iterations.")
            break
        if len(indices) < 10:
            print("Too few residues remain for alignment.")
            break
        fixed_coords = [fixed_coords[i] for i in indices]
        moving_coords = [moving_coords[i] for i in indices]
    sup.apply(moving_coords)
    return sup.rms

def align_structures_iterative(pdb1_path, pdb2_path, chain1_id='A', chain2_id='A', output_path='aligned_iterative.pdb'):
    parser = PDBParser(QUIET=True)
    s1 = parser.get_structure("ref", pdb1_path)
    s2 = parser.get_structure("mov", pdb2_path)
    c1, c2 = s1[0][chain1_id], s2[0][chain2_id]

    seq1, coords1 = extract_sequence_and_coords(c1)
    seq2, coords2 = extract_sequence_and_coords(c2)
    if not seq1 or not seq2:
        print("Empty sequences.")
        return None

    coords1_aln, coords2_aln = get_aligned_coords(seq1, coords1, seq2, coords2)
    if len(coords1_aln) < 3:
        print("Too few aligned residues.")
        return None

    rmsd = iterative_alignment(coords1_aln, coords2_aln)
    save_combined_structure(s1, s2, chain1_id, chain2_id, output_path)
    print(f"Iterative RMSD: {rmsd:.3f} Å — saved to {output_path}")
    return rmsd


In [ ]:
# --- Download PDB files ---
!wget -q https://files.rcsb.org/download/4I24.pdb -O target_raw.pdb
!wget -q https://files.rcsb.org/download/1V0O.pdb -O malaria_raw.pdb

# --- Parameters ---
target_pdb = 'target_raw.pdb'         # Experimental structure of target protein
malaria_pdb = 'malaria_raw.pdb'       # Malaria homolog structure
target_chain_id = 'A'                 # Chain in the target protein
malaria_chain_id = 'A'                # Chain in the homolog
drug_resname = '1C9'                  # Ligand residue name in target

# --- Preprocessing ---
remove_chains(target_chain_id, target_pdb, 'target_chain_only.pdb')
remove_chains(malaria_chain_id, malaria_pdb, 'malaria_chain_only.pdb')
remove_ligands('target_chain_only.pdb', drug_resname, 'target_no_ligand.pdb')

# Rename chains and ligands for consistency
rename_chains_and_ligands(
    pdb_file='malaria_chain_only.pdb',
    chain_map={malaria_chain_id: 'M'},
    ligand_map={},
    out_file='malaria_renamed.pdb'
)

rename_chains_and_ligands(
    pdb_file='target_no_ligand.pdb',
    chain_map={target_chain_id: 'T'},
    ligand_map={drug_resname: 'D'},
    out_file='target_renamed.pdb'
)


In [ ]:
# --- Alignment 1: Local sequence-based alignment ---
rmsd_local = align_structures_local(
    pdb1_path='target_renamed.pdb',
    pdb2_path='malaria_renamed.pdb',
    chain1_id='T',
    chain2_id='M',
    output_path='aligned_local.pdb'
)

# --- Alignment 2: Iterative refinement-based alignment ---
rmsd_iter = align_structures_iterative(
    pdb1_path='target_renamed.pdb',
    pdb2_path='malaria_renamed.pdb',
    chain1_id='T',
    chain2_id='M',
    output_path='aligned_iterative.pdb'
)